# 🧠 Temel Small Language Model (SLM)
### Wikipedia Verisiyle Karakter Seviyesi Dil Modeli

Bu notebook'ta sıfırdan bir Small Language Model (SLM) inşa edeceğiz.
Model, Wikipedia'dan çekilen Türkçe metin üzerinde eğitilecek ve
yeni metin üretmeyi öğrenecek.

**Mimari:** Karakter seviyesi Bigram / LSTM tabanlı dil modeli  
**Veri:** Wikipedia API (Türkçe makaleler)  
**Framework:** PyTorch

---

### 📋 İçindekiler
1. Gerekli kütüphanelerin kurulumu
2. Wikipedia'dan veri çekme
3. Veri ön işleme (tokenization)
4. Dataset ve DataLoader hazırlama
5. Model mimarisi (Bigram → LSTM)
6. Eğitim döngüsü
7. Metin üretimi
8. Sonuçlar ve değerlendirme

## 1️⃣ Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Gerekli kütüphaneleri kur
# wikipedia-api: Wikipedia makalelerini çekmek için
# torch: Sinir ağı modelini oluşturmak ve eğitmek için
!pip install wikipedia-api torch --quiet

print("✅ Kütüphaneler başarıyla kuruldu!")

In [ ]:
# Kütüphaneleri import et
import wikipediaapi          # Wikipedia'dan veri çekmek için
import torch                 # PyTorch - derin öğrenme framework'ü
import torch.nn as nn        # Sinir ağı katmanları
import torch.optim as optim  # Optimizasyon algoritmaları
from torch.utils.data import Dataset, DataLoader  # Veri yükleme
import numpy as np           # Sayısal işlemler
import matplotlib.pyplot as plt  # Görselleştirme
import random                # Rastgele işlemler
import time                  # Süre ölçümü

# Tekrar üretilebilirlik için seed sabitle
# Aynı sonuçları her çalıştırmada almak için
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU varsa kullan, yoksa CPU'ya düş
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Kullanılan cihaz: {device}")
print(f"🔢 PyTorch versiyonu: {torch.__version__}")

## 2️⃣ Wikipedia'dan Veri Çekme

Wikipedia API kullanarak Türkçe makalelerden metin toplayacağız.
Bu metinler modelimizin öğreneceği ham veriyi oluşturacak.

In [ ]:
def wikipedia_metni_cek(makale_listesi, dil='tr'):
    """
    Belirtilen Wikipedia makalelerinden metin çeker.
    
    Parametreler:
        makale_listesi (list): Çekilecek makale başlıkları
        dil (str): Wikipedia dil kodu ('tr' = Türkçe)
    
    Döndürür:
        str: Tüm makalelerin birleştirilmiş metni
    """
    # Wikipedia API'yi Türkçe dil ve kullanıcı ajanı ile başlat
    wiki = wikipediaapi.Wikipedia(
        language=dil,
        user_agent='SLM-Egitim/1.0 (egitim amacli)'  # API礼节si için gerekli
    )
    
    tum_metin = []  # Tüm makale metinlerini biriktireceğimiz liste
    
    for baslik in makale_listesi:
        print(f"📖 Çekiliyor: {baslik}...", end=' ')
        
        sayfa = wiki.page(baslik)  # Makaleyi getir
        
        if sayfa.exists():  # Makale var mı kontrol et
            metin = sayfa.text
            tum_metin.append(metin)
            print(f"✅ ({len(metin):,} karakter)")
        else:
            print(f"❌ Bulunamadı!")
    
    # Tüm metinleri boş satırla ayırarak birleştir
    return '\n\n'.join(tum_metin)


# Eğitim için kullanacağımız Wikipedia makaleleri
# Çeşitli konuları kapsayan makaleler seçiyoruz
MAKALELER = [
    'Yapay zeka',
    'Makine öğrenmesi',
    'Derin öğrenme',
    'Sinir ağı',
    'Türkiye',
    'Bilgisayar',
    'Matematik',
    'Fizik',
    'Tarih',
    'Bilim'
]

print("🌐 Wikipedia'dan veri çekiliyor...\n")
ham_metin = wikipedia_metni_cek(MAKALELER)
print(f"\n📊 Toplam metin uzunluğu: {len(ham_metin):,} karakter")

In [ ]:
# Çekilen metnin ilk 500 karakterini inceleyelim
print("📄 İlk 500 karakter:\n")
print(ham_metin[:500])
print("\n" + "="*60)
print(f"Toplam satır sayısı: {ham_metin.count(chr(10)):,}")

## 3️⃣ Veri Ön İşleme (Tokenization)

Ham metni modelin anlayabileceği sayısal formata dönüştüreceğiz.
**Karakter seviyesi tokenization** kullanacağız:
- Her karakter bir token'dır
- Karakter → Sayı eşlemesi oluştururuz
- Bu yöntem küçük modeller için idealdir

In [ ]:
def metin_on_isle(metin, max_karakter=200_000):
    """
    Ham metni temizler ve sınırlar.
    
    Parametreler:
        metin (str): İşlenecek ham metin
        max_karakter (int): Maksimum karakter sayısı
    
    Döndürür:
        str: Temizlenmiş metin
    """
    # Fazla boşlukları temizle
    import re
    metin = re.sub(r'\n{3,}', '\n\n', metin)  # 3+ boş satırı 2'ye indir
    metin = re.sub(r' {2,}', ' ', metin)        # Çoklu boşlukları tekle
    
    # Boyutu sınırla (bellek yönetimi için)
    metin = metin[:max_karakter]
    
    return metin.strip()


class KarakterTokenizer:
    """
    Karakter seviyesi tokenizer.
    
    Metindeki her benzersiz karakteri bir tamsayıya eşler.
    Bu sayede model metni sayısal vektörler olarak işleyebilir.
    
    Örnek:
        'a' → 0, 'b' → 1, 'c' → 2 ...
    """
    
    def __init__(self, metin):
        # Metindeki tüm benzersiz karakterleri bul ve sırala
        self.karakterler = sorted(set(metin))
        self.vocab_boyutu = len(self.karakterler)
        
        # Karakter → Sayı sözlüğü (encoding için)
        self.kar2idx = {kar: idx for idx, kar in enumerate(self.karakterler)}
        
        # Sayı → Karakter sözlüğü (decoding için)
        self.idx2kar = {idx: kar for idx, kar in enumerate(self.karakterler)}
    
    def encode(self, metin):
        """Metni sayı listesine çevirir."""
        return [self.kar2idx[kar] for kar in metin if kar in self.kar2idx]
    
    def decode(self, indisler):
        """Sayı listesini metne çevirir."""
        return ''.join([self.idx2kar.get(idx, '?') for idx in indisler])
    
    def __repr__(self):
        return f"KarakterTokenizer(vocab_boyutu={self.vocab_boyutu})"


# Metni ön işle
temiz_metin = metin_on_isle(ham_metin, max_karakter=150_000)
print(f"📝 İşlenmiş metin uzunluğu: {len(temiz_metin):,} karakter")

# Tokenizer'ı oluştur
tokenizer = KarakterTokenizer(temiz_metin)
print(f"🔤 Kelime hazinesi (vocab) boyutu: {tokenizer.vocab_boyutu} benzersiz karakter")
print(f"\nİlk 50 karakter: {tokenizer.karakterler[:50]}")

In [ ]:
# Tokenizer'ı test edelim
test_metni = "Yapay zeka çok güzel!"
encoded = tokenizer.encode(test_metni)
decoded = tokenizer.decode(encoded)

print(f"Orijinal : {test_metni}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")
print(f"\n✅ Encode/Decode doğru çalışıyor: {test_metni == decoded}")

# Tüm metni sayısal veriye çevir
veri = torch.tensor(tokenizer.encode(temiz_metin), dtype=torch.long)
print(f"\n📦 Tensor boyutu: {veri.shape}")
print(f"İlk 20 token: {veri[:20].tolist()}")

## 4️⃣ Dataset ve DataLoader Hazırlama

Modeli eğitmek için veriyi **eğitim/doğrulama** olarak böleceğiz.
Her örnek bir **giriş dizisi** ve ona karşılık gelen **hedef dizisinden** oluşur.

Örnek: `giriş = "Yapay zek"` → `hedef = "apay zeka"`  
Model her karakterden sonra ne geleceğini tahmin etmeyi öğrenir.

In [ ]:
class MetinDataset(Dataset):
    """
    PyTorch Dataset sınıfı.
    
    Model eğitimi için (giriş, hedef) çiftleri üretir.
    Her örnekte:
        - giriş: t anındaki karakter dizisi
        - hedef: t+1 anındaki karakter dizisi (bir sonraki karakterler)
    """
    
    def __init__(self, veri, blok_uzunlugu):
        """
        Parametreler:
            veri (Tensor): Tokenize edilmiş metin verisi
            blok_uzunlugu (int): Her örneğin sekans uzunluğu
        """
        self.veri = veri
        self.blok_uzunlugu = blok_uzunlugu
    
    def __len__(self):
        """Toplam örnek sayısını döndürür."""
        return len(self.veri) - self.blok_uzunlugu
    
    def __getitem__(self, idx):
        """
        idx'inci örneği döndürür.
        
        Giriş: [idx, idx+blok_uzunlugu) aralığındaki tokenlar
        Hedef: [idx+1, idx+blok_uzunlugu+1) aralığındaki tokenlar
        """
        giris = self.veri[idx : idx + self.blok_uzunlugu]
        hedef = self.veri[idx + 1 : idx + self.blok_uzunlugu + 1]
        return giris, hedef


# Hiperparametreler (model tasarımındaki temel konfigürasyon değerleri)
BLOK_UZUNLUGU = 64    # Her eğitim örneğinin sekans uzunluğu
BATCH_BOYUTU  = 32    # Her adımda kaç örnek işlenecek
EGITIM_ORANI  = 0.9   # Verinin %90'ı eğitim, %10'u doğrulama

# Veriyi eğitim ve doğrulama olarak böl
bolme_noktasi = int(EGITIM_ORANI * len(veri))
egitim_verisi = veri[:bolme_noktasi]
dogrulama_verisi = veri[bolme_noktasi:]

print(f"📊 Veri bölümü:")
print(f"   Eğitim   : {len(egitim_verisi):,} token ({EGITIM_ORANI*100:.0f}%)")
print(f"   Doğrulama: {len(dogrulama_verisi):,} token ({(1-EGITIM_ORANI)*100:.0f}%)")

# Dataset nesnelerini oluştur
egitim_dataset = MetinDataset(egitim_verisi, BLOK_UZUNLUGU)
dogrulama_dataset = MetinDataset(dogrulama_verisi, BLOK_UZUNLUGU)

# DataLoader: veriyi batch'ler halinde modele besler
egitim_loader = DataLoader(
    egitim_dataset,
    batch_size=BATCH_BOYUTU,
    shuffle=True,     # Her epoch'ta veriyi karıştır (genelleme için)
    drop_last=True    # Son eksik batch'i at
)
dogrulama_loader = DataLoader(
    dogrulama_dataset,
    batch_size=BATCH_BOYUTU,
    shuffle=False,    # Doğrulama verisini karıştırma
    drop_last=True
)

print(f"\n📦 Batch sayısı:")
print(f"   Eğitim   : {len(egitim_loader)} batch")
print(f"   Doğrulama: {len(dogrulama_loader)} batch")

# Bir örneği inceleyelim
giris_ornegi, hedef_ornegi = egitim_dataset[0]
print(f"\n🔍 Örnek veri:")
print(f"   Giriş shape: {giris_ornegi.shape}")
print(f"   Giriş metin: '{tokenizer.decode(giris_ornegi[:20].tolist())}...'")
print(f"   Hedef metin: '{tokenizer.decode(hedef_ornegi[:20].tolist())}...'")

## 5️⃣ Model Mimarisi

İki farklı model oluşturacağız:

### Model 1: Bigram Modeli (Basit Baseline)
- En basit dil modeli
- Sadece **önceki 1 karaktere** bakarak tahmin yapar
- Hızlı eğitim, düşük kalite

### Model 2: LSTM Modeli (Ana Model)
- **Long Short-Term Memory** ağı
- Uzun vadeli bağımlılıkları öğrenebilir
- Daha iyi metin kalitesi

In [ ]:
class BigramDilModeli(nn.Module):
    """
    Bigram Dil Modeli - En temel dil modeli.
    
    Çalışma prensibi:
        Her karakter için bir embedding vektörü öğrenir.
        Bu vektör doğrudan bir sonraki karakterin log-olasılığını verir.
        
    Kısıtlama: Sadece tek bir önceki karaktere bakar!
    Bu nedenle üretilen metin genellikle düşük kaliteli olur.
    """
    
    def __init__(self, vocab_boyutu):
        super().__init__()
        
        # Embedding tablosu: her karakter için bir satır
        # vocab_boyutu × vocab_boyutu matris
        # Her satır: o karakterden sonra hangi karakterin geleceğinin skoru
        self.embedding = nn.Embedding(vocab_boyutu, vocab_boyutu)
    
    def forward(self, x, hedefler=None):
        """
        İleri geçiş (forward pass).
        
        x: (batch, sekans) boyutlu giriş tokenları
        hedefler: (batch, sekans) boyutlu hedef tokenlar (eğitim için)
        """
        # Giriş tokenlarını embedding'e dönüştür
        # logits: (batch, sekans, vocab_boyutu)
        logitler = self.embedding(x)
        
        kayip = None
        if hedefler is not None:
            # Kaybı hesaplamak için boyutları düzenle
            # (batch*sekans, vocab_boyutu) ve (batch*sekans,)
            B, T, C = logitler.shape
            logitler_2d = logitler.view(B * T, C)
            hedefler_1d = hedefler.view(B * T)
            
            # Cross-entropy kaybı: model ne kadar yanlış tahmin yapıyor?
            kayip = nn.functional.cross_entropy(logitler_2d, hedefler_1d)
        
        return logitler, kayip
    
    def metin_uret(self, baslangic, uzunluk, tokenizer, sicaklik=1.0):
        """
        Verilen başlangıçtan metin üretir.
        
        sicaklik (temperature): 
            - Düşük (0.5): Daha tahmin edilebilir, tekrarlayan metin
            - Yüksek (2.0): Daha yaratıcı ama tutarsız metin
            - 1.0: Orijinal olasılık dağılımı
        """
        self.eval()  # Değerlendirme moduna geç (dropout vs. kapalı)
        
        with torch.no_grad():  # Gradyan hesaplamayı kapat (hız için)
            # Başlangıç metnini tokenize et
            x = torch.tensor(tokenizer.encode(baslangic), dtype=torch.long)
            x = x.unsqueeze(0).to(device)  # Batch boyutu ekle
            
            uretilen = list(tokenizer.encode(baslangic))
            
            for _ in range(uzunluk):
                # Son tokena bak (bigram: sadece 1 önceki karakter)
                logitler, _ = self(x[:, -1:])
                
                # Sıcaklık uygula
                logitler = logitler[:, -1, :] / sicaklik
                
                # Softmax: logitleri olasılığa çevir
                olasiliklar = torch.softmax(logitler, dim=-1)
                
                # Olasılık dağılımından örnekle
                sonraki_token = torch.multinomial(olasiliklar, num_samples=1)
                
                uretilen.append(sonraki_token.item())
                x = torch.cat([x, sonraki_token], dim=1)
        
        return tokenizer.decode(uretilen)


# Bigram modelini oluştur
bigram_model = BigramDilModeli(tokenizer.vocab_boyutu).to(device)

# Model parametre sayısını hesapla
param_sayisi = sum(p.numel() for p in bigram_model.parameters())
print(f"🔢 Bigram Model Parametre Sayısı: {param_sayisi:,}")

In [ ]:
class LSTMDilModeli(nn.Module):
    """
    LSTM tabanlı Dil Modeli - Ana modelimiz.
    
    Katmanlar:
        1. Embedding: Token → Vektör
        2. LSTM: Sekans bilgisini işle (uzun bağlamı hatırla)
        3. Dropout: Aşırı öğrenmeyi engelle
        4. Linear: LSTM çıkışını vocab boyutuna project et
    
    LSTM'in avantajı: Yüzlerce adım öncesindeki bilgiyi
    hafızasında tutabilir (uzun vadeli bağımlılıklar).
    """
    
    def __init__(self, vocab_boyutu, gomme_boyutu=64, gizli_boyut=128, 
                 katman_sayisi=2, dropout=0.3):
        """
        Parametreler:
            vocab_boyutu (int): Benzersiz token sayısı
            gomme_boyutu (int): Embedding vektörü boyutu
            gizli_boyut (int): LSTM gizli durum boyutu
            katman_sayisi (int): LSTM katman sayısı (derin LSTM)
            dropout (float): Dropout oranı (0-1 arası)
        """
        super().__init__()
        
        self.gizli_boyut = gizli_boyut
        self.katman_sayisi = katman_sayisi
        
        # Katman 1: Embedding
        # Her tokeni yoğun bir vektöre dönüştürür
        # Örnek: token 5 → [0.2, -0.1, 0.8, ...] (gomme_boyutu boyutlu)
        self.embedding = nn.Embedding(vocab_boyutu, gomme_boyutu)
        
        # Katman 2: LSTM
        # batch_first=True: giriş (batch, sekans, özellik) formatında
        self.lstm = nn.LSTM(
            input_size=gomme_boyutu,
            hidden_size=gizli_boyut,
            num_layers=katman_sayisi,
            batch_first=True,
            dropout=dropout if katman_sayisi > 1 else 0  # Çok katmanlıysa dropout
        )
        
        # Katman 3: Dropout (regularizasyon)
        # Eğitim sırasında nöronları rastgele sıfırlayarak aşırı öğrenmeyi engeller
        self.dropout = nn.Dropout(dropout)
        
        # Katman 4: Çıkış katmanı
        # LSTM çıkışını (gizli_boyut) vocab boyutuna map et
        self.cikis_katmani = nn.Linear(gizli_boyut, vocab_boyutu)
    
    def forward(self, x, gizli_durum=None, hedefler=None):
        """
        İleri geçiş.
        
        x: (batch, sekans) - giriş tokenları
        gizli_durum: LSTM'nin önceki gizli durumu (None ise sıfırdan başla)
        hedefler: Eğitim için hedef tokenlar
        """
        # 1. Embedding: token → vektör
        x_embedded = self.dropout(self.embedding(x))  # (batch, sekans, gomme_boyutu)
        
        # 2. LSTM: sekansı işle
        lstm_cikisi, yeni_gizli_durum = self.lstm(x_embedded, gizli_durum)
        # lstm_cikisi: (batch, sekans, gizli_boyut)
        
        # 3. Dropout uygula
        lstm_cikisi = self.dropout(lstm_cikisi)
        
        # 4. Çıkış katmanı: gizli durum → vocab skoru
        logitler = self.cikis_katmani(lstm_cikisi)  # (batch, sekans, vocab_boyutu)
        
        kayip = None
        if hedefler is not None:
            B, T, C = logitler.shape
            kayip = nn.functional.cross_entropy(
                logitler.view(B * T, C),
                hedefler.view(B * T)
            )
        
        return logitler, kayip, yeni_gizli_durum
    
    def gizli_durumu_sifirla(self, batch_boyutu):
        """Sıfır gizli durum oluşturur (yeni sekans başlangıcı için)."""
        h0 = torch.zeros(self.katman_sayisi, batch_boyutu, self.gizli_boyut).to(device)
        c0 = torch.zeros(self.katman_sayisi, batch_boyutu, self.gizli_boyut).to(device)
        return (h0, c0)
    
    def metin_uret(self, baslangic, uzunluk, tokenizer, sicaklik=0.8):
        """Verilen başlangıçtan metin üretir."""
        self.eval()
        
        with torch.no_grad():
            x = torch.tensor(tokenizer.encode(baslangic), dtype=torch.long)
            x = x.unsqueeze(0).to(device)
            
            uretilen = list(tokenizer.encode(baslangic))
            gizli = self.gizli_durumu_sifirla(1)
            
            # Önce başlangıç metnini işle
            logitler, _, gizli = self(x, gizli)
            
            for _ in range(uzunluk):
                # Son token'dan sonra ne geliyor?
                son_logit = logitler[:, -1, :] / sicaklik
                olasiliklar = torch.softmax(son_logit, dim=-1)
                sonraki = torch.multinomial(olasiliklar, num_samples=1)
                
                uretilen.append(sonraki.item())
                
                # Sadece son tokeni gir (hız için)
                logitler, _, gizli = self(sonraki, gizli)
        
        return tokenizer.decode(uretilen)


# LSTM modelini oluştur
lstm_model = LSTMDilModeli(
    vocab_boyutu=tokenizer.vocab_boyutu,
    gomme_boyutu=64,     # Embedding boyutu
    gizli_boyut=256,     # LSTM gizli durum boyutu
    katman_sayisi=2,     # 2 katmanlı LSTM
    dropout=0.3          # %30 dropout
).to(device)

param_sayisi = sum(p.numel() for p in lstm_model.parameters())
print(f"🔢 LSTM Model Parametre Sayısı: {param_sayisi:,}")
print(f"\n📐 Model mimarisi:")
print(lstm_model)

## 6️⃣ Eğitim Döngüsü

Modeli eğitmek için:
1. Veriyi modele ver
2. Kayıpı hesapla (model ne kadar yanlış?)
3. Geri yayılım (backpropagation) ile gradyanları hesapla
4. Optimizer ile ağırlıkları güncelle
5. Tekrar et!

In [ ]:
def egit(model, egitim_loader, dogrulama_loader, epoch_sayisi=10, ogrenme_hizi=0.001):
    """
    Modeli eğitir ve eğitim/doğrulama kayıplarını izler.
    
    Parametreler:
        model: Eğitilecek PyTorch modeli
        egitim_loader: Eğitim veri yükleyici
        dogrulama_loader: Doğrulama veri yükleyici
        epoch_sayisi: Kaç epoch eğitilsin
        ogrenme_hizi: Optimizer öğrenme hızı (lr)
    
    Döndürür:
        (egitim_kayiplari, dogrulama_kayiplari): Kayıp listeleri
    """
    # AdamW optimizer: Adam'ın weight decay ile geliştirilmiş versiyonu
    # L2 regularizasyon yerine weight decay kullanır → daha iyi genelleme
    optimizer = optim.AdamW(model.parameters(), lr=ogrenme_hizi, weight_decay=1e-4)
    
    # Learning rate scheduler: Eğitim ilerledikçe lr'yi azalt
    # ReduceLROnPlateau: Doğrulama kaybı iyileşmeyince lr'yi düşür
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )
    
    egitim_kayiplari = []
    dogrulama_kayiplari = []
    
    for epoch in range(epoch_sayisi):
        # ─── EĞİTİM FAZASI ───
        model.train()  # Eğitim modu (dropout aktif)
        toplam_kayip = 0
        baslangic = time.time()
        
        for batch_idx, (giris, hedef) in enumerate(egitim_loader):
            giris = giris.to(device)
            hedef = hedef.to(device)
            
            optimizer.zero_grad()  # Önceki gradyanları temizle
            
            # İleri geçiş
            if isinstance(model, LSTMDilModeli):
                _, kayip, _ = model(giris, hedefler=hedef)
            else:  # Bigram
                _, kayip = model(giris, hedef)
            
            # Geri yayılım: gradyanları hesapla
            kayip.backward()
            
            # Gradient clipping: Gradyanların patlamasını önle
            # LSTM'lerde önemli bir stabilite tekniği
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()  # Ağırlıkları güncelle
            
            toplam_kayip += kayip.item()
        
        ort_egitim_kaybi = toplam_kayip / len(egitim_loader)
        egitim_kayiplari.append(ort_egitim_kaybi)
        
        # ─── DOĞRULAMA FAZASI ───
        model.eval()  # Değerlendirme modu (dropout kapalı)
        toplam_val_kayip = 0
        
        with torch.no_grad():  # Gradyan hesaplama yok
            for giris, hedef in dogrulama_loader:
                giris = giris.to(device)
                hedef = hedef.to(device)
                
                if isinstance(model, LSTMDilModeli):
                    _, kayip, _ = model(giris, hedefler=hedef)
                else:
                    _, kayip = model(giris, hedef)
                
                toplam_val_kayip += kayip.item()
        
        ort_val_kaybi = toplam_val_kayip / len(dogrulama_loader)
        dogrulama_kayiplari.append(ort_val_kaybi)
        
        # Learning rate güncelle
        scheduler.step(ort_val_kaybi)
        
        # Perplexity: Dil modellerinin standart değerlendirme metriği
        # Düşük perplexity = daha iyi model (e^kayip)
        perplexity = np.exp(ort_val_kaybi)
        sure = time.time() - baslangic
        
        print(f"Epoch {epoch+1:2d}/{epoch_sayisi} | "
              f"Eğitim Kaybı: {ort_egitim_kaybi:.4f} | "
              f"Val Kaybı: {ort_val_kaybi:.4f} | "
              f"Perplexity: {perplexity:.1f} | "
              f"Süre: {sure:.1f}s")
    
    return egitim_kayiplari, dogrulama_kayiplari


print("🚀 Bigram modeli eğitiliyor...\n")
bigram_egitim_k, bigram_val_k = egit(
    bigram_model, egitim_loader, dogrulama_loader,
    epoch_sayisi=5,
    ogrenme_hizi=0.01
)

In [ ]:
print("🚀 LSTM modeli eğitiliyor...\n")
lstm_egitim_k, lstm_val_k = egit(
    lstm_model, egitim_loader, dogrulama_loader,
    epoch_sayisi=15,
    ogrenme_hizi=0.001
)

## 7️⃣ Sonuçları Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Eğitim Sonuçları', fontsize=14, fontweight='bold')

# Bigram kayıp grafiği
ax1 = axes[0]
ax1.plot(bigram_egitim_k, label='Eğitim Kaybı', color='steelblue', linewidth=2)
ax1.plot(bigram_val_k, label='Doğrulama Kaybı', color='coral', linewidth=2, linestyle='--')
ax1.set_title('Bigram Modeli - Kayıp Eğrisi')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Kaybı')
ax1.legend()
ax1.grid(True, alpha=0.3)

# LSTM kayıp grafiği
ax2 = axes[1]
ax2.plot(lstm_egitim_k, label='Eğitim Kaybı', color='steelblue', linewidth=2)
ax2.plot(lstm_val_k, label='Doğrulama Kaybı', color='coral', linewidth=2, linestyle='--')
ax2.set_title('LSTM Modeli - Kayıp Eğrisi')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-Entropy Kaybı')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('egitim_sonuclari.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Grafik kaydedildi: egitim_sonuclari.png")

## 8️⃣ Metin Üretimi

Eğitilen modellerle yeni metin üretelim ve karşılaştıralım!

In [ ]:
# Test başlangıç metinleri
baslangiclar = [
    "Yapay zeka",
    "Türkiye",
    "Bilim ve teknoloji"
]

print("=" * 70)
print("📝 BİGRAM MODELİ - METIN ÜRETİMİ")
print("=" * 70)

for baslangic in baslangiclar:
    uretilen = bigram_model.metin_uret(
        baslangic=baslangic,
        uzunluk=200,
        tokenizer=tokenizer,
        sicaklik=0.8
    )
    print(f"\n🔤 Başlangıç: '{baslangic}'")
    print(f"📄 Üretilen:\n{uretilen[:300]}")
    print("-" * 50)

In [ ]:
print("=" * 70)
print("🧠 LSTM MODELİ - METİN ÜRETİMİ")
print("=" * 70)

for baslangic in baslangiclar:
    uretilen = lstm_model.metin_uret(
        baslangic=baslangic,
        uzunluk=300,
        tokenizer=tokenizer,
        sicaklik=0.8
    )
    print(f"\n🔤 Başlangıç: '{baslangic}'")
    print(f"📄 Üretilen:\n{uretilen[:400]}")
    print("-" * 50)

In [ ]:
# Sıcaklık etkisini karşılaştır
print("=" * 70)
print("🌡️  SICAKLIK ETKİSİ (LSTM Modeli)")
print("=" * 70)
print("Sıcaklık, modelin ne kadar 'yaratıcı' olduğunu kontrol eder.")
print()

sicakliklar = [0.3, 0.7, 1.0, 1.5]
baslangic_metni = "Yapay zeka"

for sicaklik in sicakliklar:
    uretilen = lstm_model.metin_uret(
        baslangic=baslangic_metni,
        uzunluk=150,
        tokenizer=tokenizer,
        sicaklik=sicaklik
    )
    aciklama = {
        0.3: "(Çok tekrarlayan, tahmin edilebilir)",
        0.7: "(Dengeli)",
        1.0: "(Varsayılan)",
        1.5: "(Yaratıcı, tutarsız olabilir)"
    }
    print(f"\n🌡️  Sıcaklık={sicaklik} {aciklama[sicaklik]}")
    print(f"   {uretilen[:200]}")

## 9️⃣ Modeli Kaydet ve Değerlendir

In [ ]:
def model_kaydet(model, tokenizer, dosya_adi):
    """
    Modeli ve tokenizer'ı diske kaydeder.
    İleride yükleyip kullanabilmek için.
    """
    import pickle
    
    kayit = {
        'model_state': model.state_dict(),      # Model ağırlıkları
        'model_sinifi': model.__class__.__name__, # Model tipi
        'vocab': {
            'karakterler': tokenizer.karakterler,
            'kar2idx': tokenizer.kar2idx,
            'idx2kar': tokenizer.idx2kar,
            'vocab_boyutu': tokenizer.vocab_boyutu
        },
        'egitim_kayiplari': lstm_egitim_k,
        'dogrulama_kayiplari': lstm_val_k
    }
    
    torch.save(kayit, dosya_adi)
    print(f"✅ Model kaydedildi: {dosya_adi}")


# Modeli kaydet
model_kaydet(lstm_model, tokenizer, 'slm_model.pt')

# Final karşılaştırma tablosu
print("\n" + "=" * 60)
print("📊 FINAL MODEL KARŞILAŞTIRMASI")
print("=" * 60)
print(f"{'Metrik':<30} {'Bigram':>10} {'LSTM':>10}")
print("-" * 52)

bigram_params = sum(p.numel() for p in bigram_model.parameters())
lstm_params = sum(p.numel() for p in lstm_model.parameters())

print(f"{'Parametre Sayısı':<30} {bigram_params:>10,} {lstm_params:>10,}")
print(f"{'Son Eğitim Kaybı':<30} {bigram_egitim_k[-1]:>10.4f} {lstm_egitim_k[-1]:>10.4f}")
print(f"{'Son Val Kaybı':<30} {bigram_val_k[-1]:>10.4f} {lstm_val_k[-1]:>10.4f}")
print(f"{'Perplexity (Val)':<30} {np.exp(bigram_val_k[-1]):>10.1f} {np.exp(lstm_val_k[-1]):>10.1f}")
print("=" * 60)
print("\n💡 NOT: Düşük perplexity = daha iyi model")

## 🎯 Özet ve Sonuçlar

### Ne Öğrendik?

| Konu | Açıklama |
|------|----------|
| **Veri Toplama** | Wikipedia API ile otomatik veri çekme |
| **Tokenization** | Karakter seviyesi tokenizer implementasyonu |
| **Dataset** | PyTorch Dataset/DataLoader yapısı |
| **Bigram Modeli** | En basit dil modeli (1 önceki karakter) |
| **LSTM Modeli** | Uzun vadeli bağımlılıkları öğrenen model |
| **Eğitim** | Gradient clipping, LR scheduling |
| **Değerlendirme** | Perplexity metriği |
| **Üretim** | Sıcaklık parametresi ile kontrollü üretim |

### 🚀 Geliştirme Fikirleri

1. **Transformer Mimarisi**: LSTM yerine Attention mekanizması
2. **Daha Büyük Veri**: Daha fazla Wikipedia makalesi
3. **Word-level Tokenization**: Karakter yerine kelime tokenizasyonu
4. **Beam Search**: Greedy örnekleme yerine beam search
5. **Fine-tuning**: Belirli bir domain için ince ayar